# ln(3) Coordination Threshold — Minimal Reproducibility Notebook

**Paper:** "A topological threshold for bidirectional coordination in one-dimensional Poisson proximity networks"  
**arXiv:** 2603.15521 | **Author:** Jian Ji (jijian1@cictci.com)

This notebook reproduces **key reported numerical quantities** from public or aggregate data.  
It is a **minimal reproducibility package**, not a full raw-data replication.  
See the Summary table at the end for what requires original datasets.

---
## Contents
1. Core theorem: λℓ = ln(3)
2. Traffic blind prediction: ρ_c = 0.5093 ρ_j (MAPE = 1.9%)
3. Chengdu V2X summary statistics (aggregate only — raw data not public)
4. UTD19 Constance analysis (requires data download)
5. Regenerate Fig1 panel (a)

In [ ]:
# Install dependencies (Colab already has most of these)
!pip install numpy matplotlib scipy pandas --quiet

In [ ]:
# Environment specification
import sys, numpy, matplotlib, scipy
print(f'Python:     {sys.version.split()[0]}')
print(f'numpy:      {numpy.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'scipy:      {scipy.__version__}')
print()
print('Tested with: Python 3.10+, numpy>=1.24, matplotlib>=3.7, scipy>=1.10')
print('GitHub: https://github.com/cict001/ln3-validation')

## 1. Core Theorem: λℓ = ln(3)

In [ ]:
import numpy as np

LN3 = np.log(3)
print(f'ln(3) = {LN3:.6f}')

# Verify: unique solution to 2/(e^x - 1) = 1
x = LN3
lhs = 2 / (np.exp(x) - 1)
print(f'Verification: 2/(e^ln3 - 1) = {lhs:.6f}  (should be 1.000000)')

# Critical density ratio
def rho_c_ratio(theta):
    return (1 / (1 + theta)) ** (1 / theta)

ratio = rho_c_ratio(LN3)
print(f'\nCritical density ratio: \u03c1_c/\u03c1_j = {ratio:.6f}')
print(f'Rounded to 4 dp:        {ratio:.4f}  (paper reports 0.5093)')
print(f'Match: {abs(ratio - 0.5093) < 0.0001}')

## 2. Traffic Blind Prediction (4 datasets, MAPE = 1.9%)

In [ ]:
import numpy as np

LN3 = np.log(3)

# Published jam densities rho_j (veh/km) from original papers
# Observed critical densities from independent measurements
datasets = {
    'highD Rec.12 (DE)': {'rho_j': 80,  'rho_c_obs': 40.7},
    'NGSIM I-80 (US)':   {'rho_j': 70,  'rho_c_obs': 35.6},
    'Zen Traffic (JP)':  {'rho_j': 95,  'rho_c_obs': 50.0},
    'pNEUMA (GR)':       {'rho_j': 60,  'rho_c_obs': 29.4},
}
# Note: Chengdu excluded — no independently measured rho_c available from OBU data

def rho_c_ratio(theta):
    return (1 / (1 + theta)) ** (1 / theta)

print('Pre-registered blind prediction: \u03c1_c = 0.5093 \u00d7 \u03c1_j')
print(f'{"Dataset":<22} {"\u03c1_j":>6} {"Pred":>7} {"Obs":>7} {"Error":>7}')
print('-' * 55)

errors = []
for name, d in datasets.items():
    pred = rho_c_ratio(LN3) * d['rho_j']
    err  = abs(pred - d['rho_c_obs']) / d['rho_c_obs'] * 100
    errors.append(err)
    print(f'{name:<22} {d["rho_j"]:>6.0f} {pred:>7.1f} {d["rho_c_obs"]:>7.1f} {err:>6.1f}%')

mape = np.mean(errors)
print('-' * 55)
print(f'{"MAPE":<22} {"":>6} {"":>7} {"":>7} {mape:>6.1f}%')
print(f'\nPaper reports MAPE = 1.9% \u2014 reproduced: {mape:.1f}%')

## 3. Chengdu V2X — Aggregate Statistics

> **Note:** Raw data is not publicly available (CICT/Chengdu institutional data).  
> This cell verifies internal consistency of aggregate statistics reported in the paper.  
> Contact jijian1@cictci.com for academic verification requests.

In [ ]:
import numpy as np

print('=== Chengdu V2X Aggregate Statistics ===')
print(f'Total records:     N = 19,782,736 OBU records')
print(f'Speed bimodal:     41.7% stopped (<5 km/h), 20.8% free-flow (>60 km/h)')
print(f'Mean speed:        30.8 km/h')
print(f'Gap CV:            2.09  (Poisson reference = 1.000)')
print()

cv = 2.09
print(f'Gap CV = {cv:.2f} >> 1.000 (Poisson)')
print(f'Over-dispersed spacing consistent with sub-threshold urban flow (\u03bb\u2113 < ln(3))')
print()

# Cluster lifetime analysis
# Exact values: 7.14 and 1.795 minutes
# Paper rounds to 7.1 and 1.8; ratio = 3.98x
print('=== Cluster Lifetime Analysis ===')
print(f'Total cluster events: 779,023')
print(f'Definition: N >= 3 vehicles within 300m simultaneously')
mean_super = 7.14    # exact value (paper: 7.1 min)
mean_sub   = 1.795   # exact value (paper: 1.8 min)
ratio = mean_super / mean_sub
print(f'Super-threshold mean: {mean_super} min  (paper: 7.1 min)')
print(f'Sub-threshold mean:   {mean_sub} min (paper: 1.8 min)')
print(f'Ratio:                {ratio:.2f}\u00d7  \u2714 matches paper 3.98\u00d7')
print(f'p-value:              < 10^-4 (Mann-Whitney one-tailed)')

## 4. UTD19 Constance Analysis
*(Requires data download from https://utd19.ethz.ch)*

In [ ]:
# Full UTD19 replication:
# 1. Download from https://utd19.ethz.ch (DOI: 10.1038/s41597-019-0001-9, ~6.5 GB)
# 2. Expected directory structure:
#      UTD19/
#        utd19_u.csv        (detector metadata with lat/lon)
#        data/constance/    (loop detector time series)
# 3. Run:
#      python analysis/analyze_UTD19_final.py --data_dir /path/to/UTD19
# 4. Expected output columns:
#      detector_id, variance_sub, variance_super, variance_ratio, p_value, significant
# 5. Should identify 5 significant detectors:
#      K33.D4.1  ratio=2.66  p=0.0005
#      K20D4.11  ratio=2.55  p<0.0001
#      Z33       ratio=1.92  p=0.003
#      K21D2.1   ratio=1.66  p=0.035
#      K51D3.1   ratio=1.61  p=0.038

# Demonstrate the statistical logic without data:
import numpy as np
from scipy import stats

np.random.seed(42)
sub = np.random.normal(60, 18, 200)   # night: sub-threshold (high variance)
sup = np.random.normal(25,  8, 200)   # day congested: super-threshold (low variance)

var_ratio = np.var(sub) / np.var(sup)
stat, pval = stats.mannwhitneyu(
    np.abs(sub - np.mean(sub)),
    np.abs(sup - np.mean(sup)),
    alternative='greater'
)
print(f'Simulated variance ratio (sub/super): {var_ratio:.2f}\u00d7')
print(f'Mann-Whitney p-value: {pval:.4f}')
print(f'\nFor real UTD19 data:')
print(f'  88 detectors with diurnal \u03bb\u2113 crossing ln(3)')
print(f'  5 show statistically significant variance reduction (p < 0.05)')
print(f'  83 non-significant: gradual crossing / mixed traffic / insufficient N')
print(f'  (83 non-significant is NOT evidence against the threshold)')

## 5. Regenerate Fig1 Panel (a)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

LN3 = np.log(3)

def rho_c_ratio(theta):
    return (1 / (1 + theta)) ** (1 / theta)

datasets = {
    'highD (DE)':  (80,  40.7, 'highway'),
    'NGSIM (US)':  (70,  35.6, 'highway'),
    'Zen (JP)':    (95,  50.0, 'urban'),
    'pNEUMA (GR)': (60,  29.4, 'urban'),
}

fig, ax = plt.subplots(figsize=(5, 5))
rng = np.linspace(25, 55, 100)
ax.fill_between(rng, rng*0.95, rng*1.05, alpha=0.15, color='grey', label='\u00b15% band')
ax.plot(rng, rng, 'k--', lw=1, label='Perfect agreement')

for name, (rj, rc_obs, typ) in datasets.items():
    rc_pred = rho_c_ratio(LN3) * rj
    color = '#1f77b4' if typ == 'highway' else '#d62728'
    marker = 'o' if typ == 'highway' else '^'
    ax.scatter(rc_obs, rc_pred, color=color, marker=marker, s=80, zorder=5)
    ax.annotate(name.split(' ')[0], (rc_obs, rc_pred),
                textcoords='offset points', xytext=(5, 3), fontsize=9)

errors = [abs(rho_c_ratio(LN3)*rj - rc) / rc * 100 for rj, rc, _ in datasets.values()]
mape = np.mean(errors)

ax.set_xlabel('Observed \u03c1_c (veh/km)', fontsize=11)
ax.set_ylabel('Predicted \u03c1_c = 0.5093 \u03c1_j (veh/km)', fontsize=11)
ax.set_title(f'Blind prediction (MAPE = {mape:.1f}%)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig1a_blind_prediction.pdf', bbox_inches='tight')
plt.show()
print(f'MAPE = {mape:.1f}%  (paper reports 1.9%)')

---
## Summary

| Result | Paper | Reproduced |
|--------|-------|------------|
| ln(3) = unique solution to 2/(e^x-1)=1 | ✓ | ✓ |
| ρ_c/ρ_j = 0.5093 | ✓ | ✓ |
| Traffic MAPE = 1.9% (4 datasets) | ✓ | ✓ |
| Gap CV = 2.09 (sub-threshold) | ✓ | Aggregate only |
| Cluster lifetime 3.98× | ✓ | Aggregate only |
| UTD19 variance ratios 1.61–2.66 | ✓ | Needs UTD19 data |

**Reproducibility scope:**  
This notebook provides *minimal reproducibility* — key quantities verifiable from public or aggregate data.  
For full raw-data replication, see `analysis/` scripts in the GitHub repository.

**Data requirements:**
- Cells 1–3: No data needed (theory + published aggregates)
- Cell 4: Requires UTD19 download (~6.5 GB) from https://utd19.ethz.ch
- highD/NGSIM: Requires dataset registration at respective sites
- Chengdu V2X: Not publicly available (CICT institutional data)